In [1]:
# install mujoco + robosuite
!apt-get install -y libglew-dev python-opengl ffmpeg
!pip install robosuite
!pip install mujoco
!apt-get update -y
!apt-get install -y ffmpeg
!pip install robosuite imageio

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Reading package lists... Done
E: Could not open lock file /var/lib/apt/lists/lock - open (13: Permission denied)
E: Unable to lock directory /var/lib/apt/lists/
W: Problem unlinking the file /var/cache/apt/pkgcache.bin - RemoveCaches (13: Permission denied)
W: Problem unlinking the file /var/cache/apt/srcpkgcache.bin - RemoveCaches (13: Permission denied)
E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
Defaulting to user installation because normal site-packages is not writeable


In [ ]:
pip install torch

In [2]:
# Cell 1: Imports and utils (fully fixed)

import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.normal import Normal

from torchvision import transforms
from IPython.display import display, Video

import imageio

# Ensure this notebook directory is in sys.path so utils.py can be imported
project_dir = Path().resolve()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

print("Project directory:", project_dir)
print("Contents:", [p.name for p in project_dir.iterdir()])

# Only import functions that actually exist
from utils import (
    make_noisy_lift_env,
    init_frames_dir,
    is_lift_success,   # <-- added
)


Project directory: /U1/accounts/jkapit1/Documents/Deep learning
Contents: ['frames_rl_corrected', 'eval_run_2', 'controller_rl_corrected_demo.mp4', 'eval_run_3', 'Robosuite_Demo_2.ipynb', 'Robosuite_Demo.ipynb', 'best_policy_demo.mp4', 'eval_run_7', 'eval_run_4', 'eval_run_1', 'controller_demo.mp4', 'eval_run_10', 'eval_run_8', 'eval_run_9', 'eval_run_5', 'DLS-3.pdf', 'utils.py', 'Robosuite Demo2.ipynb', 'utils2.py', '__pycache__', 'eval_run_6', 'frames']


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /U1/accounts/jkapit1/.local/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


In [3]:
# Cell 1b: Define create_image() fallback

from PIL import Image

def create_image(env, width=256, height=256, camera="frontview"):
    """
    Returns a PIL image captured from the specified camera.
    """
    frame = env.sim.render(
        camera_name=camera,
        width=width,
        height=height,
    )
    # frame is (H, W, 3) RGB uint8
    return Image.fromarray(frame)


In [4]:
# Cell 1c (fixed): Define save_video() using ffmpeg CLI, not imageio

def save_video(frames_dir, output_filename, fps=20):
    """
    Given a directory of frame_XXXX.png images, turn them into an MP4 video
    using the ffmpeg command-line tool.
    """
    # Make sure the directory exists and has frames
    frame_files = sorted([
        f for f in os.listdir(frames_dir)
        if f.lower().endswith(".png")
    ])
    if not frame_files:
        print(f"[save_video] No PNG frames found in {frames_dir}. Not creating video.")
        return None

    # Build ffmpeg command. Assumes frames are named frame_0000.png, frame_0001.png, ...
    cmd = (
        f"ffmpeg -y -framerate {fps} "
        f"-i {frames_dir}/frame_%04d.png "
        f"-c:v libx264 -pix_fmt yuv420p {output_filename}"
    )
    print("[save_video] Running:", cmd)
    os.system(cmd)

    if os.path.exists(output_filename):
        print(f"[save_video] Video saved to {output_filename}")
        return output_filename
    else:
        print(f"[save_video] Failed to create video {output_filename}")
        return None


In [5]:
# Cell 2: Policy network definition

class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional branch for image (3 x 64 x 64)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2),  # 16 x 32 x 32
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2), # 32 x 16 x 16
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2), # 64 x 8 x 8
            nn.ReLU(),
            nn.Flatten(),                                          # 64 * 8 * 8 = 4096
        )

        cnn_out_dim = 64 * 8 * 8

        # MLP branch for noisy cube position (3D)
        self.mlp_pos = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

        # Combined head
        self.fc = nn.Sequential(
            nn.Linear(cnn_out_dim + 64, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
        )

        # Output: mean and log_std for 2D correction (x, y)
        self.mean_head = nn.Linear(64, 2)
        self.log_std_head = nn.Linear(64, 2)

    def forward(self, img, noisy_pos):
        """
        img:  (B, 3, 64, 64)
        noisy_pos: (B, 3)
        """
        img_feat = self.cnn(img)
        pos_feat = self.mlp_pos(noisy_pos)
        x = torch.cat([img_feat, pos_feat], dim=-1)
        x = self.fc(x)

        mean = self.mean_head(x)
        log_std = self.log_std_head(x)
        log_std = torch.clamp(log_std, -5, 2)  # keep std reasonable

        return mean, log_std


In [6]:
# Cell 3: Helper to sample actions and compute log-prob

def select_action(policy, img_tensor, pos_tensor):
    """
    img_tensor: (1, 3, 64, 64)
    pos_tensor: (1, 3)
    Returns:
      action (np array, shape (2,)),
      log_prob (torch scalar)
    """
    mean, log_std = policy(img_tensor, pos_tensor)
    std = torch.exp(log_std)

    dist = Normal(mean, std)
    action = dist.sample()
    log_prob = dist.log_prob(action).sum(dim=-1)  # sum over action dims

    return action.detach().cpu().numpy()[0], log_prob


In [7]:
# Cell 4: Movement-based shaped reward (closer -> +, farther -> -)

def compute_step_reward(obs, prev_dist_xy=None):
    """
    Reward design:
      - Positive reward if the gripper moves closer to the cube in XY.
      - Negative reward if it moves farther away.
      - Small success bonus if the cube is lifted (encourages lifting).
    """
    gripper_pos = obs.get("robot0_eef_pos", None)   # (3,)
    cube_pos_true = obs.get("cube_pos", None)       # (3,)

    # If we can't access positions, no shaping
    if gripper_pos is None or cube_pos_true is None:
        return 0.0, prev_dist_xy

    gripper_pos = np.array(gripper_pos, dtype=float)
    cube_pos_true = np.array(cube_pos_true, dtype=float)

    # Distance in XY plane
    gripper_xy = gripper_pos[:2]
    cube_xy = cube_pos_true[:2]
    curr_dist_xy = np.linalg.norm(gripper_xy - cube_xy)

    # Movement reward: compare previous distance to current
    if prev_dist_xy is None:
        move_reward = 0.0
    else:
        delta = prev_dist_xy - curr_dist_xy  # >0 if we got closer
        move_reward = 5.0 * delta            # scale factor

    # Success / height bonus
    cube_height = cube_pos_true[2]
    success_bonus = 0.0
    if cube_height > 0.10:  # lifted 10 cm
        success_bonus = 5.0

    # Small time penalty to encourage faster solutions (per step)
    time_penalty = -0.001

    shaped = move_reward + success_bonus + time_penalty
    return float(shaped), curr_dist_xy


In [8]:
# Cell 5: Run one episode with shaped reward (for training)

def run_episode(policy, env, device, max_steps=150, gamma=0.99):
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])

    obs = env.reset()
    rewards = []
    log_probs = []

    prev_dist_xy = None

    for t in range(max_steps):
        img = obs["frontview_image"]
        noisy_pos = obs["cube_pos_noisy"]

        # Prepare tensors
        img_tensor = transform(img).unsqueeze(0).to(device)
        pos_tensor = torch.FloatTensor(noisy_pos).unsqueeze(0).to(device)

        # Get 2D correction from policy
        action_xy, log_prob = select_action(policy, img_tensor, pos_tensor)

        # Build full 7D action
        action = np.zeros(env.action_dim, dtype=np.float32)
        action[:2] = np.clip(action_xy, -1.0, 1.0)

        obs, env_reward, done, info = env.step(action)

        # Movement-based shaped reward
        r, prev_dist_xy = compute_step_reward(obs, prev_dist_xy)
        rewards.append(r)
        log_probs.append(log_prob)

        if done:
            break

    # Compute discounted returns
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    if len(returns) > 1:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8 + 1e-12)

    log_probs = torch.stack(log_probs)

    return log_probs, returns, sum(rewards)


In [9]:
# Cell 6: Training loop (REINFORCE) with shaped reward

def train_agent(num_episodes=250, gamma=0.99, lr=1e-4, max_steps=150):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy = PolicyNet().to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    episode_returns = []

    for ep in range(1, num_episodes + 1):
        env = make_noisy_lift_env(add_noise=True, image_size=(256, 256))

        log_probs, returns, total_reward = run_episode(
            policy, env, device, max_steps=max_steps, gamma=gamma
        )

        # Policy gradient loss
        loss = -(log_probs * returns).sum()

        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)

        optimizer.step()

        episode_returns.append(total_reward)

        if ep % 10 == 0:
            avg_return = np.mean(episode_returns[-10:])
            print(f"Episode {ep:4d} | Last 10 avg shaped return: {avg_return:.3f}")

    return policy


In [10]:
# Cell 7: Evaluation and video recording (best episode by shaped return)

def evaluate_and_record(policy, num_tests=10, max_steps=150, video_name="best_policy_demo"):
    device = next(policy.parameters()).device

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])

    runs = []   # store info for each episode

    print(f"\nEvaluating on {num_tests} random environments...")

    for i in range(num_tests):
        env = make_noisy_lift_env(add_noise=True, image_size=(256, 256))

        # Unique frames dir for this episode
        frames_dir = f"eval_run_{i+1}"
        init_frames_dir(frames_dir)

        obs = env.reset()
        cube_start_pos = np.asarray(obs["cube_pos_noisy"], dtype=float).copy()

        prev_dist_xy = None
        info = {}
        total_reward = 0.0

        for t in range(max_steps):
            img = obs["frontview_image"]
            noisy_pos = obs["cube_pos_noisy"]

            img_tensor = transform(img).unsqueeze(0).to(device)
            pos_tensor = torch.FloatTensor(noisy_pos).unsqueeze(0).to(device)

            with torch.no_grad():
                mean, log_std = policy(img_tensor, pos_tensor)
            correction_xy = mean.cpu().numpy()[0]  # deterministic action at eval

            # Build full action vector of correct dimension
            action = np.zeros(env.action_dim, dtype=np.float32)
            action[:2] = np.clip(correction_xy, -1.0, 1.0)

            obs, env_reward, done, info = env.step(action)

            # Save frame
            frame_img = create_image(env)  # PIL image
            frame_path = os.path.join(frames_dir, f"frame_{t:04d}.png")
            frame_img.save(frame_path)

            # Shaped movement reward
            r, prev_dist_xy = compute_step_reward(obs, prev_dist_xy)
            total_reward += r

            if done:
                break

        # Define success: use info["success"] if available, otherwise use is_lift_success
        if "success" in info:
            success = bool(info["success"])
        else:
            success = is_lift_success(
                obs,
                cube_start_pos=cube_start_pos,
                min_lift=0.10,
                max_xy_shift=0.10,
                use_noisy=True,
            )

        runs.append({
            "idx": i + 1,
            "frames_dir": frames_dir,
            "success": success,
            "total_reward": total_reward,
        })

        print(f"Test {i+1}: Success={success}, shaped return={total_reward:.2f}")

    # ---- Summary stats: success rate + average shaped return ----
    success_count = sum(r["success"] for r in runs)
    avg_return = np.mean([r["total_reward"] for r in runs]) if runs else 0.0

    print("-" * 50)
    print(f"Final Success Rate (eval): {success_count}/{num_tests} ({100.0 * success_count / num_tests:.1f}%)")
    print(f"Average shaped return (eval): {avg_return:.2f}")

    # ---- Choose best episode for video ----
    # First, see if there is any successful run
    successful_runs = [r for r in runs if r["success"]]
    if len(successful_runs) > 0:
        # Choose the successful run with highest total_reward
        best_run = max(successful_runs, key=lambda r: r["total_reward"])
        print(f"Using BEST SUCCESSFUL episode for video: episode {best_run['idx']}, return={best_run['total_reward']:.2f}")
    else:
        # No successes: choose the run with highest shaped return
        best_run = max(runs, key=lambda r: r["total_reward"])
        print(f"No successful episodes. Using BEST ATTEMPT: episode {best_run['idx']}, return={best_run['total_reward']:.2f}")

    chosen_dir = best_run["frames_dir"]

    # Always create a video from the chosen episode
    video_file = f"{video_name}.mp4"
    video_file = save_video(chosen_dir, video_file, fps=20)  # use return value


    return video_file, success_count, avg_return


In [11]:
# Cell 8: Train, evaluate, display video, and report final stats

trained_policy = train_agent(num_episodes=250, gamma=0.99, lr=1e-4, max_steps=150)

video_file, success_count, avg_return = evaluate_and_record(
    trained_policy,
    num_tests=10,
    max_steps=150,
    video_name="best_policy_demo",
)

# Show video from the best episode
if isinstance(video_file, str) and os.path.exists(video_file):
    display(Video(video_file, embed=True, width=640))
else:
    print("Video file not found:", video_file)

# Final metrics
final_accuracy = 100.0 * success_count / 10.0
print("=" * 60)
print(f"FINAL ACCURACY: {final_accuracy:.1f}% ({success_count}/10)")
print(f"FINAL AVERAGE SHAPED RETURN (eval): {avg_return:.2f}")


[robosuite INFO] Loading controller configuration from: /U1/accounts/jkapit1/.local/lib/python3.10/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the contro

Episode   10 | Last 10 avg shaped return: 748.905


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

Episode   20 | Last 10 avg shaped return: 749.370


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

...

Test 10: Success=False, shaped return=748.29
--------------------------------------------------
Final Success Rate (eval): 0/10 (0.0%)
Average shaped return (eval): 748.40
No successful episodes. Using BEST ATTEMPT: episode 7, return=748.55
[save_video] Running: ffmpeg -y -framerate 20 -i eval_run_7/frame_%04d.png -c:v libx264 -pix_fmt yuv420p best_policy_demo.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

[save_video] Video saved to best_policy_demo.mp4


frame=  150 fps=0.0 q=-1.0 Lsize=      47kB time=00:00:07.35 bitrate=  52.5kbits/s speed=19.5x    
video:45kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 5.833315%
[libx264 @ 0x55b071c6f040] frame I:1     Avg QP:15.49  size:  7543
[libx264 @ 0x55b071c6f040] frame P:38    Avg QP:19.01  size:   667
[libx264 @ 0x55b071c6f040] frame B:111   Avg QP:27.84  size:   108
[libx264 @ 0x55b071c6f040] consecutive B-frames:  1.3%  0.0%  0.0% 98.7%
[libx264 @ 0x55b071c6f040] mb I  I16..4:  6.2% 51.2% 42.6%
[libx264 @ 0x55b071c6f040] mb P  I16..4:  0.0%  0.0%  0.0%  P16..4:  5.2%  3.7%  3.7%  0.0%  0.0%    skip:87.4%
[libx264 @ 0x55b071c6f040] mb B  I16..4:  0.0%  0.0%  0.0%  B16..8:  3.1%  0.8%  0.4%  direct: 1.3%  skip:94.4%  L0:38.9% L1:40.9% BI:20.2%
[libx264 @ 0x55b071c6f040] 8x8 transform intra:50.8% inter:12.1%
[libx264 @ 0x55b071c6f040] coded y,uvDC,uvAC intra: 84.5% 76.4% 55.0% inter: 2.5% 0.4% 0.0%
[libx264 @ 0x55b071c6f040] i16 v,h,dc,p:  6% 75%  0% 19%
[lib

FINAL ACCURACY: 0.0% (0/10)
FINAL AVERAGE SHAPED RETURN (eval): 748.40


In [12]:
# Cell 9a: Numeric evaluation – success rate and average shaped reward

def evaluate_policy(policy, num_episodes=10, max_steps=150):
    device = next(policy.parameters()).device
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])

    success_count = 0
    episode_returns = []

    for ep in range(1, num_episodes + 1):
        env = make_noisy_lift_env(add_noise=True, image_size=(256, 256))
        obs = env.reset()
        cube_start_pos = np.asarray(obs["cube_pos_noisy"], dtype=float).copy()

        total_reward = 0.0
        prev_dist_xy = None
        info = {}

        for t in range(max_steps):
            img = obs["frontview_image"]
            noisy_pos = obs["cube_pos_noisy"]

            img_tensor = transform(img).unsqueeze(0).to(device)
            pos_tensor = torch.FloatTensor(noisy_pos).unsqueeze(0).to(device)

            with torch.no_grad():
                mean, log_std = policy(img_tensor, pos_tensor)
            correction_xy = mean.cpu().numpy()[0]

            action = np.zeros(env.action_dim, dtype=np.float32)
            action[:2] = np.clip(correction_xy, -1.0, 1.0)

            obs, env_reward, done, info = env.step(action)

            r, prev_dist_xy = compute_step_reward(obs, prev_dist_xy)
            total_reward += r

            if done:
                break

        # Success using is_lift_success
        success = is_lift_success(
            obs,
            cube_start_pos=cube_start_pos,
            min_lift=0.10,
            max_xy_shift=0.10,
            use_noisy=True,
        )
        if success:
            success_count += 1

        episode_returns.append(total_reward)
        print(f"Episode {ep}: success={success}, shaped return={total_reward:.2f}")

    accuracy = 100.0 * success_count / num_episodes
    avg_return = float(np.mean(episode_returns))

    print("=" * 60)
    print(f"Final accuracy: {accuracy:.1f}% ({success_count}/{num_episodes})")
    print(f"Final average shaped return: {avg_return:.2f}")

    return accuracy, avg_return


# Run numeric evaluation (after training)
final_accuracy, final_avg_reward = evaluate_policy(trained_policy, num_episodes=10, max_steps=150)


[robosuite INFO] Loading controller configuration from: /U1/accounts/jkapit1/.local/lib/python3.10/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

Episode 1: success=False, shaped return=748.49


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

Episode 2: success=False, shaped return=748.30


...

Episode 8: success=False, shaped return=748.43


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

Episode 9: success=False, shaped return=748.53


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

Episode 10: success=False, shaped return=748.27
Final accuracy: 0.0% (0/10)
Final average shaped return: 748.39


In [13]:
# Cell 9b: Test RL+controller on 10 envs and record video of best episode

from utils import (
    init_frames_dir,
    save_frame,
    step_with_action,
    move_ee_to,
    is_lift_success,
)

from IPython.display import Video as IPyVideo

def run_rl_corrected_episode(trained_policy, run_idx, max_steps_per_phase=None):
    """
    Runs ONE episode of the RL-corrected scripted controller:
      1) Use RL to correct noisy cube position once at the start.
      2) Use scripted phases (above -> down -> close -> lift -> hold).
      3) Save frames into a run-specific frames_dir.
      4) Return success flag, final cube height, and frames_dir.
    """
    if max_steps_per_phase is None:
        # Just for reference; the actual steps are defined in move_ee_to / step_with_action calls
        max_steps_per_phase = {
            "open": 20,
            "above": 100,
            "down": 250,
            "close": 40,
            "lift": 200,
            "hold": 40,
        }

    # --- Create fresh noisy environment ---
    env = make_noisy_lift_env(add_noise=True, image_size=(256, 256))
    obs = env.reset()

    cam_name = "frontview"
    cam_key = f"{cam_name}_image"
    assert cam_key in obs, (
        f"Missing {cam_key} in obs. Make sure use_camera_obs=True and "
        f"camera_names includes '{cam_name}'."
    )

    # Starting cube position (noisy) for success evaluation
    cube_start_pos = np.asarray(obs["cube_pos_noisy"], dtype=float).copy()

    H, W = obs[cam_key].shape[:2]

    # --- Use RL policy to correct the noisy cube position (single-shot) ---
    device = next(trained_policy.parameters()).device
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])

    img = obs[cam_key]                 # (H, W, 3)
    noisy_pos = obs["cube_pos_noisy"]  # (3,)

    img_tensor = transform(img).unsqueeze(0).to(device)          # (1, 3, 64, 64)
    pos_tensor = torch.FloatTensor(noisy_pos).unsqueeze(0).to(device)  # (1, 3)

    with torch.no_grad():
        mean, log_std = trained_policy(img_tensor, pos_tensor)
    correction_xy = mean.cpu().numpy()[0]

    # Build corrected cube position from policy output
    cube_pos_meas = np.asarray(noisy_pos, dtype=float).copy()
    cube_pos_corr = cube_pos_meas.copy()
    cube_pos_corr[:2] = cube_pos_meas[:2] + correction_xy  # apply correction in x,y

    print(f"[Run {run_idx}] Noisy cube pos:     {cube_pos_meas}")
    print(f"[Run {run_idx}] Corrected cube pos: {cube_pos_corr}")

    # --- Prepare / clear frame directory for this run ---
    frames_dir = f"frames_run_{run_idx}"
    init_frames_dir(frames_dir)

    action_dim = env.action_dim
    print(f"[Run {run_idx}] action_dim =", action_dim)

    frame_id = 0

    # --- Save first frame ---
    frame_id = save_frame(obs, cam_key, frame_id, frames_dir=frames_dir)

    # --- Waypoint offsets in world frame ---
    above_height = 0.15  # 15 cm above cube
    grasp_height = 0.02  # (optional) fine adjustment
    lift_height = 0.25   # lift height after grasp

    # --- Build waypoints from the RL-corrected cube position ---
    target_above = cube_pos_corr.copy()
    target_above[2] += above_height

    target_grasp = cube_pos_corr.copy()
    # target_grasp[2] += grasp_height  # enable if you want to offset the grasp height

    target_lift = cube_pos_corr.copy()
    target_lift[2] += lift_height

    # ----------------------------------------------------------------
    # Scripted policy: Above → Down → Close → Lift (using RL-corrected target)
    # ----------------------------------------------------------------

    print(f"[Run {run_idx}] Phase 0: open gripper")
    action_open = np.zeros(action_dim, dtype=float)
    action_open[-1] = -1.0
    obs, frame_id = step_with_action(
        env,
        action_open,
        n_steps=20,
        obs=obs,
        cam_key=cam_key,
        frame_id=frame_id,
        frames_dir=frames_dir,
    )

    print(f"[Run {run_idx}] Phase 1: move above cube (corrected target)")
    obs, frame_id = move_ee_to(
        env,
        obs,
        target_pos_or_fn=target_above,
        gripper=-1.0,
        steps=100,
        action_dim=action_dim,
        cam_key=cam_key,
        frame_id=frame_id,
        frames_dir=frames_dir,
        kp=8.0,
        ki=0.0,
        kd=1.0,
        max_delta=0.1,
    )

    print(f"[Run {run_idx}] Phase 2: move down to grasp height (corrected target)")
    obs, frame_id = move_ee_to(
        env,
        obs,
        target_pos_or_fn=target_grasp,
        gripper=-1.0,
        steps=250,
        action_dim=action_dim,
        cam_key=cam_key,
        frame_id=frame_id,
        frames_dir=frames_dir,
        kp=[10.0, 10.0, 12.0],
        ki=0.0,
        kd=0.2,
        max_delta=0.1,
    )

    print(f"[Run {run_idx}] Phase 3: close gripper")
    action_close = np.zeros(action_dim, dtype=float)
    action_close[-1] = 1.0
    obs, frame_id = step_with_action(
        env,
        action_close,
        n_steps=40,
        obs=obs,
        cam_key=cam_key,
        frame_id=frame_id,
        frames_dir=frames_dir,
    )

    print(f"[Run {run_idx}] Phase 4: lift cube (corrected target)")
    obs, frame_id = move_ee_to(
        env,
        obs,
        target_pos_or_fn=target_lift,
        gripper=1.0,
        steps=200,
        action_dim=action_dim,
        cam_key=cam_key,
        frame_id=frame_id,
        frames_dir=frames_dir,
        kp=10.0,
        ki=0.0,
        kd=1.0,
        max_delta=0.1,
    )

    print(f"[Run {run_idx}] Phase 5: hold")
    obs, frame_id = step_with_action(
        env,
        np.zeros(action_dim, dtype=float),
        n_steps=40,
        obs=obs,
        cam_key=cam_key,
        frame_id=frame_id,
        frames_dir=frames_dir,
    )

    print(f"[Run {run_idx}] RL-corrected scripted rollout finished. Frames saved to {frames_dir}")

    # --- Success check ---
    success = is_lift_success(
        obs,
        cube_start_pos=cube_start_pos,
        min_lift=0.10,     # require 10 cm lift
        max_xy_shift=0.10, # allow up to 10 cm XY drift
        use_noisy=True,
    )

    # Final cube height (noisy) as a "how good" metric
    final_cube_height = float(np.asarray(obs["cube_pos_noisy"], dtype=float)[2])

    print(f"[Run {run_idx}] Lift success: {success}, final cube height (noisy): {final_cube_height:.3f}")

    return success, final_cube_height, frames_dir


def test_controller_on_10_envs(trained_policy, num_envs=10, video_name="best_result_demo.mp4"):
    """
    Runs the RL-corrected scripted controller on N=10 environments
    with random object locations and noise, and:
      - Computes success rate.
      - Picks the best episode (prefer success; otherwise highest final cube height).
      - Creates a video for that episode.
    """
    results = []

    for i in range(1, num_envs + 1):
        print("=" * 60)
        print(f"Running RL-corrected controller on environment {i}/{num_envs}")
        success, final_height, frames_dir = run_rl_corrected_episode(trained_policy, run_idx=i)
        results.append({
            "idx": i,
            "success": success,
            "final_height": final_height,
            "frames_dir": frames_dir,
        })

    # Compute success rate
    success_count = sum(1 for r in results if r["success"])
    success_rate = 100.0 * success_count / num_envs
    print("=" * 60)
    print(f"Success rate over {num_envs} environments: {success_rate:.1f}% ({success_count}/{num_envs})")

    # Choose best episode:
    #   1) Among successful ones: highest final cube height.
    #   2) If no success: highest final cube height overall.
    successful_runs = [r for r in results if r["success"]]
    if successful_runs:
        best_run = max(successful_runs, key=lambda r: r["final_height"])
        print(f"Best run (SUCCESS) is env {best_run['idx']} with final height {best_run['final_height']:.3f}")
    else:
        best_run = max(results, key=lambda r: r["final_height"])
        print(f"No successful runs. Best ATTEMPT is env {best_run['idx']} with final height {best_run['final_height']:.3f}")

    best_frames_dir = best_run["frames_dir"]

    # Create video from best run's frames
    video_file = video_name
    video_file = save_video(best_frames_dir, video_file, fps=20)

    if isinstance(video_file, str) and os.path.exists(video_file):
        print(f"Best result video saved to {video_file}")
        display(IPyVideo(video_file, embed=True, width=640))
    else:
        print("Failed to create best result video. video_file =", video_file)

    return success_rate, results, video_file


# ---- Run the test (after training) ----
# trained_policy should already exist from your training cell.
success_rate, test_results, best_video = test_controller_on_10_envs(trained_policy, num_envs=10, video_name="best_result_demo.mp4")


[robosuite INFO] Loading controller configuration from: /U1/accounts/jkapit1/.local/lib/python3.10/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


Running RL-corrected controller on environment 1/10


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

[Run 1] Noisy cube pos:     [0.00762903 0.02579417 0.83147311]
[Run 1] Corrected cube pos: [-0.07231634 -0.13556207  0.83147311]
[Run 1] action_dim = 7
[Run 1] Phase 0: open gripper
[Run 1] Phase 1: move above cube (corrected target)
[Run 1] Phase 2: move down to grasp height (corrected target)
[Run 1] Phase 3: close gripper
[Run 1] Phase 4: lift cube (corrected target)
[Run 1] Phase 5: hold


[robosuite INFO] Loading controller configuration from: /U1/accounts/jkapit1/.local/lib/python3.10/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


...

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

[Run 10] Noisy cube pos:     [3.19508312e-04 4.00156491e-02 8.31504464e-01]
[Run 10] Corrected cube pos: [-0.07970593 -0.1211647   0.83150446]
[Run 10] action_dim = 7
[Run 10] Phase 0: open gripper
[Run 10] Phase 1: move above cube (corrected target)
[Run 10] Phase 2: move down to grasp height (corrected target)
[Run 10] Phase 3: close gripper
[Run 10] Phase 4: lift cube (corrected target)
[Run 10] Phase 5: hold
[Run 10] RL-corrected scripted rollout finished. Frames saved to frames_run_10
[Run 10] Lift success: False, final cube height (noisy): 0.821
Success rate over 10 environments: 0.0% (0/10)
No successful runs. Best ATTEMPT is env 9 with final height 0.822
[save_video] Running: ffmpeg -y -framerate 20 -i frames_run_9/frame_%04d.png -c:v libx264 -pix_fmt yuv420p best_result_demo.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

[save_video] Video saved to best_result_demo.mp4
Best result video saved to best_result_demo.mp4


frame=  651 fps=274 q=-1.0 Lsize=     144kB time=00:00:32.40 bitrate=  36.3kbits/s speed=13.6x    
video:135kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 6.257223%
[libx264 @ 0x55a7b8cf1080] frame I:3     Avg QP:13.29  size:  9779
[libx264 @ 0x55a7b8cf1080] frame P:165   Avg QP:19.22  size:   469
[libx264 @ 0x55a7b8cf1080] frame B:483   Avg QP:25.41  size:    64
[libx264 @ 0x55a7b8cf1080] consecutive B-frames:  0.6%  1.2%  0.5% 97.7%
[libx264 @ 0x55a7b8cf1080] mb I  I16..4:  4.2% 54.9% 40.9%
[libx264 @ 0x55a7b8cf1080] mb P  I16..4:  0.0%  0.0%  0.0%  P16..4:  4.7%  2.7%  2.2%  0.0%  0.0%    skip:90.3%
[libx264 @ 0x55a7b8cf1080] mb B  I16..4:  0.0%  0.0%  0.0%  B16..8:  1.6%  0.4%  0.2%  direct: 0.9%  skip:96.9%  L0:42.2% L1:41.7% BI:16.1%
[libx264 @ 0x55a7b8cf1080] 8x8 transform intra:55.2% inter:11.5%
[libx264 @ 0x55a7b8cf1080] coded y,uvDC,uvAC intra: 88.7% 84.8% 69.2% inter: 1.6% 0.3% 0.0%
[libx264 @ 0x55a7b8cf1080] i16 v,h,dc,p:  6% 72% 12%  9%
[li